In [1]:
import pandas as pd
import numpy as np

In [2]:
movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

In [3]:
movie_stats = ratings.groupby("movieId").agg(
    avg_rating=("rating", "mean"),
    num_ratings=("rating", "count")
).reset_index()

movie_stats.head()

,movieId,avg_rating,num_ratings
0,1,3.897438,68997
1,2,3.275758,28904
2,3,3.139447,13134
3,4,2.845331,2806
4,5,3.059602,13154


In [4]:
movie_stats = movie_stats.merge(
    movies,
    on="movieId"
)

movie_stats.head()

,movieId,avg_rating,num_ratings,title,genres
0,1,3.897438,68997,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,3.275758,28904,Jumanji (1995),Adventure|Children|Fantasy
2,3,3.139447,13134,Grumpier Old Men (1995),Comedy|Romance
3,4,2.845331,2806,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,3.059602,13154,Father of the Bride Part II (1995),Comedy


In [5]:
movie_stats.sort_values(
    "avg_rating",
    ascending=False
).head(20)

,movieId,avg_rating,num_ratings,title,genres
81273,282081,5.0,1,Shadows (2023),Thriller
81271,282073,5.0,1,Mal de Ojo (2022),Horror
81251,282021,5.0,1,Mr. Krueger's Christmas (1980),Children|Drama
81250,282019,5.0,1,It's a Wonderful Life (2007),Comedy|Fantasy|Romance
54558,193529,5.0,2,Traces of Smoke (1992),(no genres listed)
54572,193557,5.0,1,Love Pret-a-porte (2017),Comedy|Romance
54814,194066,5.0,1,The Revolution That Wasn't (2008),(no genres listed)
54654,193727,5.0,1,Still Burning (2016),Drama
81226,281956,5.0,1,ManFish (2022),Comedy|Horror
81199,281902,5.0,1,Going Varsity In Mariachi,Documentary


In [6]:
C = movie_stats["avg_rating"].mean()
C

np.float64(3.005082384974345)

In [7]:
m = movie_stats["num_ratings"].quantile(0.90)
m

np.float64(248.90000000000873)

In [8]:
movie_stats["score"] = (
    (movie_stats["num_ratings"] /
    (movie_stats["num_ratings"] + m))
    * movie_stats["avg_rating"]
) + (
    (m /
    (movie_stats["num_ratings"] + m))
    * C
)

In [9]:
movie_stats = movie_stats[
    movie_stats["num_ratings"] >= m
].copy()

print(f"Movies qualifying for recommendation: {len(movie_stats)}")

Movies qualifying for recommendation: 8444


In [10]:
top_movies = movie_stats.sort_values(
    "score",
    ascending=False
)

top_movies[
    ["title", "avg_rating", "num_ratings", "score"]
].head(20)

,title,avg_rating,num_ratings,score
314,"Shawshank Redemption, The (1994)",4.404614,102929,4.401238
39306,Planet Earth (2006),4.444369,2948,4.332311
840,"Godfather, The (1972)",4.317030,66440,4.312134
44042,Band of Brothers (2001),4.426539,2811,4.310914
58698,Parasite (2019),4.312254,11670,4.284956
44190,Planet Earth II (2016),4.446830,1956,4.284079
49,"Usual Suspects, The (1995)",4.265070,67750,4.260458
1190,"Godfather: Part II, The (1974)",4.264468,43111,4.257239
1173,12 Angry Men (1957),4.265311,21863,4.251126
522,Schindler's List (1993),4.236990,73849,4.232852


In [11]:
def get_top_movies(n=10):
    return (
        top_movies
        .sort_values("score", ascending=False)
        [["title", "avg_rating", "num_ratings", "score"]]
        .head(n)
    )

In [12]:
get_top_movies(10)

,title,avg_rating,num_ratings,score
314,"Shawshank Redemption, The (1994)",4.404614,102929,4.401238
39306,Planet Earth (2006),4.444369,2948,4.332311
840,"Godfather, The (1972)",4.317030,66440,4.312134
44042,Band of Brothers (2001),4.426539,2811,4.310914
58698,Parasite (2019),4.312254,11670,4.284956
44190,Planet Earth II (2016),4.446830,1956,4.284079
49,"Usual Suspects, The (1995)",4.265070,67750,4.260458
1190,"Godfather: Part II, The (1974)",4.264468,43111,4.257239
1173,12 Angry Men (1957),4.265311,21863,4.251126
522,Schindler's List (1993),4.236990,73849,4.232852


In [13]:
top_movies.to_csv("../outputs/top_movies.csv", index=False)